# R4-G2 - Línea base `g2_baseline_v1`: clasificación binaria de lesiones ováricas (mapeo v2, partición `phash_le8_v1`)

**Objetivo.** Orquestar, por subproceso verificado, la ejecución del protocolo G2: tres arquitecturas (ResNet50, DenseNet121, ConvNeXt-Tiny) × tres semillas (42, 43, 44) × cinco folds = 45 corridas, para lesión no maligna (categorías 0, 1, 2, 3, 4, 6) frente a lesión maligna (categoría 7); la categoría 5 está excluida.

**Estado.** Partición provisional; conformidad académica pendiente. Este notebook **no autoriza entrenar**: `RUN_TRAINING = False` por defecto y solo el autor lo habilita, en el entorno autorizado (Falcon), tras la auditoría independiente y las precondiciones del protocolo §4.7. No usa el test oficial ni las 180 normales.

## Configuración y fuente del protocolo

- Protocolo congelado: `src/r4_v2/protocolo_g2_baseline_v1.md` (`sha256_contenido_blob` registrado en la configuración).
- Configuración: `configs/r4_v2/g2_baseline_v1.yaml`. Partición: `configs/splits/v2_auto/cv_folds_phash_le8_v1.csv`, verificada por `sha256_bytes_archivo` (K1); transferencia privada, no versionada.
- Código: `src/r4_v2/` (modos `referencia` → `humo` → `completa`, protocolo §4.7).
- Resultados: `results/r4_v2/phash_le8_v1/g2_baseline_v1/` (nunca bajo `results/classification/`).

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

RUN_TRAINING = False          # el autor lo habilita explícitamente; aplica a cualquier modo del ejecutor
RUN_REPORTE = False           # genera T1-T6 y F1-F4 desde predicciones guardadas (solo tras las 45 corridas)
MODO = "referencia"           # "referencia" -> "humo" -> "completa" (orden fijo del protocolo §4.7)
NOTEBOOK = "notebooks/R4_v2/01_g2_baseline_v1.ipynb"

# Atestaciones del autor: se registran en run_config; el código no las verifica ni las otorga.
AUTORIZACION_EJECUCION = ""
AUDITORIA_INDEPENDIENTE = ""
CONFORMIDAD_ASESOR = ""       # solo humo/completa; debe constar como real

def localizar_raiz() -> Path:
    for candidato in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidato / "src" / "r4_v2").is_dir():
            return candidato
    raise FileNotFoundError("No se pudo localizar la raíz del repositorio.")

PROJECT_ROOT = localizar_raiz()

## Método

El ejecutor se lanza **por subproceso** con `-B` y `-X pycache_prefix=<directorio nuevo, vacío y fuera del repositorio>` (protocolo §4.5.6); no se ejecuta en el proceso del kernel. Antes de cada corrida verifica K1-K8 y la integridad §4.5 (protocolo, configuración, código ejecutado y este notebook, que debe estar versionado y sin cambios). Sin lógica de entrenamiento ni de selección aquí.

In [ ]:
prefijo_pyc = tempfile.mkdtemp(prefix="r4_pyc_")   # fuera del repositorio y vacío
cmd = [sys.executable, "-B", "-X", f"pycache_prefix={prefijo_pyc}", "-m", "src.r4_v2.ejecutar_g2",
       "--modo", MODO, "--notebook", NOTEBOOK,
       "--autorizacion-ejecucion", AUTORIZACION_EJECUCION, "--auditoria-independiente", AUDITORIA_INDEPENDIENTE,
       "--conformidad-asesor", CONFORMIDAD_ASESOR]
if RUN_TRAINING:
    resultado = subprocess.run(cmd, cwd=PROJECT_ROOT)
    resultado.check_returncode()
else:
    print("RUN_TRAINING = False: no se ejecuta nada.")

## Verificación y reporte

Reporte determinista (T1-T6, F1-F4) a partir de las predicciones guardadas de las 45 corridas `completa`; exige K7 y escribe en un directorio nuevo. Humo y referencia no entran en T1-T5 ni F1-F4.

In [ ]:
destino_reporte = "results/r4_v2/phash_le8_v1/g2_baseline_v1/reporte"
cmd_reporte = [sys.executable, "-B", "-X", f"pycache_prefix={tempfile.mkdtemp(prefix='r4_pyc_')}", "-m", "src.r4_v2.reporte",
               "--salida", destino_reporte]
if RUN_REPORTE:
    subprocess.run(cmd_reporte, cwd=PROJECT_ROOT).check_returncode()
else:
    print("RUN_REPORTE = False: no se ejecuta nada.")

## Limitaciones

- La partición `phash_le8_v1` es provisional: pHash ≤ 8 es un criterio exploratorio y post hoc, sin revisión humana; no equivale a separación por paciente y deja riesgo residual de similitudes no agrupadas.
- 38 positivas en total (7-8 por fold): las métricas por fold son descriptivas y los IC del bootstrap agrupado no cubren la variabilidad entre semillas, de partición ni de selección.
- Checkpoint y parada se eligen con el mismo fold que se evalúa: el AP OOF es optimista en magnitud desconocida.
- La clase maligna contiene un subtipo (categoría 7) del dataset; no se demuestra generalización a otros cánceres ováricos.
- Este notebook solo orquesta; no hay resultados experimentales todavía.

## Interpretación

*Pendiente.* Se escribe solo después de disponer de resultados realmente ejecutados, identificando mapeo v2, `phash_le8_v1` y `g2_baseline_v1`.